# SCB 2D simulation figures

Figure code for the manuscript. All analysis functions are provided by the
`scb2d_analysis` package this notebook is part of; the notebook itself loads the
data and assembles the panels.

| Module | Contents |
| --- | --- |
| `datasets` | input paths and the assignment of simulations to conditions |
| `io_simulation` | readers for the three raw simulation output formats |
| `monoclonality` | statistics for P(Monoclonal \| Visible) |
| `fixation` | fixation and extinction probability curves over time |
| `replacement` | replacement probability of a labeled stem cell |
| `experimental_data` | loader for the measured mouse data |
| `plot_style` | palette, line styles, shared matplotlib configuration |
| `plots` | plotting functions, each drawing into the current axes |

The sensitivity of the results to the replacement rate $k_r$ is examined in
`plot_influence_of_kr_publication.ipynb`.

To run this notebook on another machine, adjust the roots at the top of
`datasets.py`, or set `SCB2D_SIMULATION_ROOT` and `SCB2D_DOWNLOAD_ROOT`. The
check in the following cell reports missing inputs.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The notebook is located inside the package, therefore the directory
# *containing* `scb2d_analysis` has to be on the import path. It is located by
# walking up from the working directory.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "scb2d_analysis" / "__init__.py").exists():
        if str(folder) not in sys.path:
            sys.path.insert(0, str(folder))
        break

from scb2d_analysis import datasets, experimental_data, fixation, io_simulation
from scb2d_analysis import monoclonality, plots, replacement
from scb2d_analysis.plot_style import (
    CM, COLORS, LINE_STYLES, PAPER_RED,
    P_MONOCLONAL_GIVEN_VISIBLE_LABEL, save_figure, setup_plot_style,
)

plt.rcdefaults()

# Font of the panels. matplotlib reads these settings when a figure is created,
# so they are applied here rather than in the figure cells; a `setup_plot_style`
# call after `plt.subplots` would not reach the ticks of that figure.
plt.rcParams.update({"font.size": 9, "font.family": "Arial"})

# Export of the panels. Set to True to write every figure into FIGURE_DIR;
# `run_notebooks.py` turns it on through the environment.
SAVE_FIGURES = os.environ.get("SCB2D_SAVE_FIGURES") == "1"
FIGURE_DIR = Path(datasets.__file__).parent / "figures"

# Report configured but unreachable inputs before any analysis is run.
missing = datasets.missing_paths()
if missing:
    raise FileNotFoundError("configured but missing:\n  " + "\n  ".join(missing))
print("all input paths found")

## Experimental data

Measured percentage of fixed crypts per condition, read from the Excel sheet.
The keys are `block<N>_<condition>`, following the block layout of the sheet.

The measured data are optional. If the sheet is not available on this machine,
`measured_condition` returns `(None, None)` and the panels below are drawn from
the simulation alone.

In [ ]:
measured = experimental_data.load_experimental_data_if_available(
    datasets.EXPERIMENTAL_DATA_FILE
)
experiment_days = np.array(datasets.EXPERIMENT_DAYS)


def measured_condition(key):
    """
    Return (averages, standard deviations) of one condition, in percent.

    Returns (None, None) if the experimental data are not available, so that the
    figures can be drawn from the simulation alone.
    """
    if key not in measured:
        return None, None

    return np.array(measured[key]["Avg"]), np.array(measured[key]["SD"])


# Proximal colon
wt_prox_colon_avg, wt_prox_colon_sd = measured_condition("block1_WT")
braf_prox_colon_avg, braf_prox_colon_sd = measured_condition("block1_BRAF")

# MHCII
mhcii_braf_fl_avg, mhcii_braf_fl_sd = measured_condition("block2_BRAF; MHCII fl/+")
mhcii_braf_fl_fl_avg, mhcii_braf_fl_fl_sd = measured_condition("block2_BRAF; MHCII fl/fl")

# IL10RA
braf_il_fl_avg, braf_il_fl_sd = measured_condition("block3_BRAF; IL10RA fl/+")
braf_il_fl_fl_avg, braf_il_fl_fl_sd = measured_condition("block3_BRAF; IL10RA fl/fl")

# Diphtheria toxin
braf_dt_control_avg, braf_dt_control_sd = measured_condition("block4_BRAF")
braf_dt_avg, braf_dt_sd = measured_condition("block4_BRAF+DT")

# anti-IL10RA antibody
braf_anti_il_control_avg, braf_anti_il_control_sd = measured_condition("block5_BRAF")
braf_anti_il_avg, braf_anti_il_sd = measured_condition("block5_BRAF+anti-IL10RA")

if measured:
    print(f"{len(measured)} experimental conditions loaded")
else:
    print("no experimental data available; the figures are drawn without the "
          "measured data points")

## Figure: P(Monoclonal | Visible) over time

Simulated curves (mean ± 1 SD across repetitions) against the measured data
points of WT and BRAF in the proximal colon.

In [ ]:
plt.figure(figsize=(8, 5))
setup_plot_style(font_size=9)

# Measured data points, drawn only if the experimental data are available.
if braf_prox_colon_avg is not None:
    plots.plot_experimental_points(experiment_days,
                                   braf_prox_colon_avg, braf_prox_colon_sd, PAPER_RED)
if wt_prox_colon_avg is not None:
    plots.plot_experimental_points(experiment_days,
                                   wt_prox_colon_avg, wt_prox_colon_sd, COLORS[0])

# Legend proxies combining the marker of the data with the line of the simulation.
plt.plot([], [], linestyle=LINE_STYLES[0], marker="o", color=PAPER_RED, label=r"$\text{BRAF}$")
plt.plot([], [], linestyle=LINE_STYLES[1], marker="o", color=COLORS[0], label=r"$\text{WT}$")

# Simulated curves; the empty label excludes them from the legend.
simulation_styles = {
    "BRAF": (LINE_STYLES[0], PAPER_RED),
    "WT": (LINE_STYLES[1], COLORS[0]),
}
for condition, (line_style, color) in simulation_styles.items():
    probabilities_by_day = io_simulation.read_visible_monoclonal_probabilities_from_directory(
        datasets.PERCENT_FIXED_FOLDERS[condition]
    )
    mean_by_day, std_by_day = monoclonality.average_probability_per_day(probabilities_by_day)
    plots.plot_probability_over_days(mean_by_day, std_by_day, line_style, color, "")

plt.ylabel(P_MONOCLONAL_GIVEN_VISIBLE_LABEL)
plt.xlabel("Days")
plt.legend(loc="upper left", frameon=False)
plt.xlim(0, 25)
plt.ylim(0, 1)

if SAVE_FIGURES:
    save_figure("p_monoclonal_visible_wt_braf", FIGURE_DIR)

plt.show()

## Fixation and extinction over time

`fixation_events[condition]` holds, for every crypt of every simulation run, the
day it became monoclonal and the founder lineage that took it over. Reading the
folders is time-consuming and is therefore done once here and reused by both
figures below.

In [ ]:
fixation_events = {
    condition: io_simulation.read_fixation_events_from_directory(folder)
    for condition, folder in datasets.FIXATION_FOLDERS.items()
}

for condition, runs in fixation_events.items():
    print(f"{condition}: {len(runs)} simulation runs")

# Styling shared by the two figures below.
condition_styles = {
    "BRAF": (PAPER_RED, LINE_STYLES[0]),
    "WT": (COLORS[0], LINE_STYLES[1]),
}

### Figure: fixation probability

Cumulative probability that a crypt has been taken over by a *labeled* lineage,
averaged over simulation runs.

In [ ]:
LAST_PLOTTED_DAY = 60

plt.subplots(figsize=(7.65 * CM, 5.83 * CM), constrained_layout=True)
setup_plot_style(font_size=9)

for condition, (color, line_style) in condition_styles.items():
    events_by_day = fixation.group_fixation_events_by_day(
        fixation_events[condition], max_day=LAST_PLOTTED_DAY
    )
    curves = fixation.compute_cumulative_probability_per_run(events_by_day)

    days, means, stds = fixation.average_probability_curves(curves)
    _, _, _, confidence_intervals = fixation.average_probability_curves_with_ci(curves)
    print(f"{condition} - "
          f"{fixation.format_probability_at_day(days, means, stds, confidence_intervals, LAST_PLOTTED_DAY)}")

    plots.plot_mean_with_std_band(days, means, stds, color, color, condition, line_style)

plt.ylim(0, 1)
plt.xlim(0, LAST_PLOTTED_DAY)
plt.ylabel("Fixation Probability")
plt.legend(loc="center right", bbox_to_anchor=(1, 0.35), frameon=False)

if SAVE_FIGURES:
    save_figure("fixation_probability_wt_braf", FIGURE_DIR)

### Figure: extinction probability

Cumulative probability that all labeled lineages of a crypt died out, i.e. that
the crypt was taken over by an unlabeled cell. Computed up to day 101, so that
the value at the end of the simulated period can be reported, and plotted up to
day 60.

In [ ]:
REPORTED_DAY = 101

plt.subplots(figsize=(10.03 * CM, 7.01 * CM), constrained_layout=True)
setup_plot_style(font_size=9)

for condition, (color, line_style) in condition_styles.items():
    events_by_day = fixation.group_extinction_events_by_day(
        fixation_events[condition], max_day=REPORTED_DAY
    )
    curves = fixation.compute_cumulative_probability_per_run(events_by_day)

    days, means, stds = fixation.average_probability_curves(curves)
    _, _, _, confidence_intervals = fixation.average_probability_curves_with_ci(curves)
    print(f"{condition} - "
          f"{fixation.format_probability_at_day(days, means, stds, confidence_intervals, REPORTED_DAY)}")

    plots.plot_mean_with_std_band(days, means, stds, color, color, condition, line_style)

plt.ylim(0, 1)
plt.xlim(0, 60)
plt.ylabel("Extinction Probability")
plots.reversed_legend(loc="lower right", bbox_to_anchor=(1, 0.2), frameon=False)

if SAVE_FIGURES:
    save_figure("extinction_probability_wt_braf", FIGURE_DIR)

## Replacement probability

Fraction of stem cell replacements caused by a labeled cell, per crypt. Only
crypts that became monoclonal are used. A value of 0.5 indicates that labeled
and unlabeled cells compete neutrally.

The runs are the independent units, therefore every crypt of a run is averaged
first and the statistics are computed over those run means.

In [ ]:
replacement_summaries = {}

for condition in datasets.REPORTED_REPLACEMENT_CONDITIONS:
    dataset = datasets.REPLACEMENT_DATASETS[condition]

    counts = io_simulation.read_replacement_counts_from_directory(dataset["path"])
    _, probabilities_monoclonal = replacement.compute_replacement_probabilities(counts)

    (run_means, mean_of_run_means, std_of_run_means,
     ci_of_run_means, *_) = replacement.summarize_run_means_with_ci(probabilities_monoclonal)

    replacement_summaries[condition] = {
        "run_means": run_means,
        "mean": mean_of_run_means,
        "std": std_of_run_means,
        "ci": ci_of_run_means,
        "parameters": dataset["parameters"],
    }

    print(f"{condition} ({dataset['parameters']})")
    print("  " + replacement.format_run_mean_summary(
        mean_of_run_means, std_of_run_means, ci_of_run_means))

### Figure: replacement probability distribution, WT vs mutant

Density histogram of the per-run replacement probabilities. The dashed line at
0.5 marks neutral competition.

`HISTOGRAM_CONDITIONS` selects the two conditions of the panel; the mutant run
shown is the one closest to the kd = 0.53 run of the earlier figure version.

In [ ]:
plt.subplots(figsize=(7.65 * CM, 5.82 * CM), constrained_layout=True)

# Condition of `datasets.REPLACEMENT_DATASETS` -> (color, legend entry).
HISTOGRAM_CONDITIONS = {
    "WT": (COLORS[0], "WT"),
    "LI 0-1, 0.0625 / 0.25, mutant kd = 0.5": (PAPER_RED, "BRAF"),
}

for condition, (color, label) in HISTOGRAM_CONDITIONS.items():
    summary = replacement_summaries[condition]
    print(f"{condition}: mean {summary['mean']:.3f} ± {summary['std']:.3f}")
    plots.plot_replacement_probability_histogram(
        summary["run_means"], color, label=label,
        num_bins=20, mark_neutral_line=True,
    )

plots.reversed_legend(loc="upper left", bbox_to_anchor=(-0.01, 1.00),
                      frameon=False, handletextpad=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

if SAVE_FIGURES:
    save_figure("replacement_probability_wt_mutant", FIGURE_DIR)